# 02 · Preprocesamiento — Sofascore

En este notebook se aplica un preprocesamiento técnico mínimo a los datos crudos de **Sofascore**.
El objetivo es únicamente dejar los datos bien estructurados, sin tomar aún decisiones analíticas
(nulos, outliers, selección de variables), que se abordarán en fases posteriores.

**Acciones realizadas:**
1. Renombrado de columnas con espacios en el nombre
2. Conversión del tipo de dato de `data_season`
3. Reordenación de columnas (identificadoras → stats clave → resto alfabético)

**Estructura de archivos:**
- Entrada:  `data/raw/sofascore/df_<liga>_<temporada>.csv` (36 archivos)
- Salida:   `data/processed/sofascore/clean_<liga>_<temporada>.csv` (36 archivos)

---

## 1. Imports y configuración de rutas

In [1]:
import pandas as pd
from pathlib import Path

# Ruta raíz del proyecto (dos niveles arriba desde notebooks/02_preprocesamiento/)
ROOT = Path.cwd().parents[1]

# Rutas base del proyecto
RAW_DIR       = ROOT / 'data' / 'raw' / 'sofascore'
PROCESSED_DIR = ROOT / 'data' / 'processed' / 'sofascore'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print('✅ Rutas configuradas')
print(f'   Root:      {ROOT}')
print(f'   Raw:       {RAW_DIR}')
print(f'   Processed: {PROCESSED_DIR}')

✅ Rutas configuradas
   Root:      d:\USER\Desktop\TFM
   Raw:       d:\USER\Desktop\TFM\data\raw\sofascore
   Processed: d:\USER\Desktop\TFM\data\processed\sofascore


---
## 2. Prototipo sobre `df_spain_2425.csv`

Desarrollamos y validamos la lógica de preprocesamiento sobre un único archivo piloto
antes de generalizarla al resto.

### 2.1 Carga e inspección inicial

In [2]:
df_proto = pd.read_csv(RAW_DIR / 'df_spain_2425.csv')

print(f'Dimensiones: {df_proto.shape[0]} filas x {df_proto.shape[1]} columnas')
df_proto.head(5)

Dimensiones: 589 filas x 116 columnas


,accurateChippedPasses,accurateCrosses,accurateCrossesPercentage,accurateFinalThirdPasses,accurateLongBalls,accurateLongBallsPercentage,accurateOppositionHalfPasses,accurateOwnHalfPasses,accuratePasses,accuratePassesPercentage,...,touches,wasFouled,yellowCards,yellowRedCards,player,team,player id,team id,data_country,data_season
0,3,0,0.00,3,19,54.29,13,12,25,60.98,...,53,0,0,0,Aitor Fernández,Osasuna,99516,2820,spain,2425
1,8,0,0.00,11,63,38.41,40,79,119,53.60,...,299,1,0,0,Leo Román,Mallorca,1131909,2826,spain,2425
2,62,53,24.65,511,46,58.97,838,209,994,79.46,...,1993,37,4,0,Raphinha,Barcelona,831005,2817,spain,2425
3,45,22,20.37,633,30,48.39,980,158,1116,78.93,...,2361,60,3,0,Lamine Yamal,Barcelona,1402912,2817,spain,2425
4,49,6,17.14,573,45,78.95,800,139,933,85.28,...,1734,40,3,0,Kylian Mbappé,Real Madrid,826643,2829,spain,2425


### 2.2 Renombrado de columnas

Las columnas `player id` y `team id` contienen espacios, lo que puede causar problemas
en operaciones posteriores (joins, acceso por atributo, etc.). Se renombran a `player_id` y `team_id`.

In [3]:
df_proto = df_proto.rename(columns={
    'player id': 'player_id',
    'team id'  : 'team_id'
})

print('✅ Renombradas: player id → player_id | team id → team_id')

✅ Renombradas: player id → player_id | team id → team_id


### 2.3 Corrección de tipos de datos

`data_season` se almacena como entero (ej. `2425`). Al ser un identificador de temporada
y no una magnitud numérica, se convierte a `string`.

In [4]:
df_proto['data_season'] = df_proto['data_season'].astype(str)

print(f'✅ data_season convertida a str  →  {df_proto["data_season"].unique()}')

✅ data_season convertida a str  →  ['2425']


### 2.4 Reordenación de columnas

Se establece el siguiente criterio de ordenación:
1. **Columnas identificadoras**: identifican unívocamente al jugador, equipo, liga y temporada
2. **Stats clave**: las variables más relevantes para el análisis (minutos, rating, goles, asistencias, xG, xA)
3. **Resto de stats**: orden alfabético

In [5]:
# Columnas identificadoras
cols_id = ['player', 'player_id', 'team', 'team_id', 'data_country', 'data_season']

# Stats más relevantes para el análisis
cols_clave = [
    'minutesPlayed', 'appearances', 'matchesStarted',
    'rating', 'goals', 'assists', 'goalsAssistsSum',
    'expectedGoals', 'expectedAssists'
]

# Resto de columnas estadísticas en orden alfabético
cols_resto = sorted([
    c for c in df_proto.columns
    if c not in cols_id and c not in cols_clave
])

orden_final = cols_id + cols_clave + cols_resto
df_proto = df_proto[orden_final]

print(f'✅ Columnas reordenadas: {len(cols_id)} id | {len(cols_clave)} clave | {len(cols_resto)} resto')
print(f'   Total: {len(orden_final)} columnas')
df_proto.head(5)

✅ Columnas reordenadas: 6 id | 9 clave | 101 resto
   Total: 116 columnas


,player,player_id,team,team_id,data_country,data_season,minutesPlayed,appearances,matchesStarted,rating,...,totalOppositionHalfPasses,totalOwnHalfPasses,totalPasses,totalRating,totalShots,totwAppearances,touches,wasFouled,yellowCards,yellowRedCards
0,Aitor Fernández,99516,Osasuna,2820,spain,2425,90,1,1,8.40,...,27,14,41,8.4,0,1,53,0,0,0
1,Leo Román,1131909,Mallorca,2826,spain,2425,630,7,7,7.83,...,126,96,222,54.8,0,3,299,1,0,0
2,Raphinha,831005,Barcelona,2817,spain,2425,2844,36,32,7.80,...,1224,242,1251,280.7,114,14,1993,37,4,0
3,Lamine Yamal,1402912,Barcelona,2817,spain,2425,2861,35,31,7.79,...,1330,192,1414,272.8,144,11,2361,60,3,0
4,Kylian Mbappé,826643,Real Madrid,2829,spain,2425,2915,34,34,7.70,...,976,153,1094,261.7,161,12,1734,40,3,0


### 2.5 Validación del prototipo

In [6]:
assert list(df_proto.columns[:6]) == cols_id,   'ERROR: las primeras columnas no son las identificadoras'
assert 'player_id' in df_proto.columns,         'ERROR: falta columna player_id'
assert 'player id' not in df_proto.columns,     'ERROR: sigue existiendo player id con espacio'
assert df_proto['data_season'].dtype == object, 'ERROR: data_season no es str'

print('✅ Todas las validaciones superadas')
print(f'   Dimensiones finales: {df_proto.shape}')

✅ Todas las validaciones superadas
   Dimensiones finales: (589, 116)


---
## 3. Función generalizada `limpiar_sofascore()`

Se encapsula toda la lógica anterior en una función reutilizable para aplicarla
de forma consistente sobre los 36 archivos.

In [7]:
# Orden de columnas definido una única vez como constante
COLS_ID = ['player', 'player_id', 'team', 'team_id', 'data_country', 'data_season']

COLS_CLAVE = [
    'minutesPlayed', 'appearances', 'matchesStarted',
    'rating', 'goals', 'assists', 'goalsAssistsSum',
    'expectedGoals', 'expectedAssists'
]


def limpiar_sofascore(df: pd.DataFrame) -> pd.DataFrame:
    """
    Aplica el preprocesamiento técnico estándar a un DataFrame crudo de Sofascore.

    Pasos:
      1. Renombra columnas con espacios (player id, team id).
      2. Convierte data_season a string.
      3. Reordena columnas: identificadoras -> stats clave -> resto alfabético.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame crudo leído directamente del CSV de raw.

    Returns
    -------
    pd.DataFrame
        DataFrame preprocesado y listo para la fase de procesamiento (03).
    """
    df = df.copy()

    # 1. Renombrar columnas con espacios
    df = df.rename(columns={'player id': 'player_id', 'team id': 'team_id'})

    # 2. Convertir data_season a string
    df['data_season'] = df['data_season'].astype(str)

    # 3. Reordenar columnas
    cols_resto = sorted([c for c in df.columns if c not in COLS_ID and c not in COLS_CLAVE])
    df = df[COLS_ID + COLS_CLAVE + cols_resto]

    return df


print('✅ Función limpiar_sofascore() definida')

✅ Función limpiar_sofascore() definida


---
## 4. Aplicación sobre los 36 archivos

Se procesan todos los CSVs de `data/raw/sofascore/` y se guardan los resultados
en `data/processed/sofascore/` con el mismo nombre.

In [8]:
archivos_raw = sorted(RAW_DIR.glob('*.csv'))
print(f'Archivos encontrados: {len(archivos_raw)}')
print()

resumen = []

for archivo in archivos_raw:
    try:
        df_raw   = pd.read_csv(archivo)
        df_clean = limpiar_sofascore(df_raw)

        nombre_salida = archivo.name
        df_clean.to_csv(PROCESSED_DIR / nombre_salida, index=False)

        resumen.append({
            'archivo' : archivo.name,
            'filas'   : len(df_clean),
            'columnas': df_clean.shape[1],
            'estado'  : '✅'
        })

    except Exception as e:
        resumen.append({
            'archivo' : archivo.name,
            'filas'   : None,
            'columnas': None,
            'estado'  : f'❌ {e}'
        })

df_resumen = pd.DataFrame(resumen)
print(df_resumen.to_string(index=False))
print()
print(f'Procesados correctamente: {(df_resumen["estado"] == "✅").sum()} / {len(df_resumen)}')

Archivos encontrados: 36

                              archivo  filas  columnas estado
                  df_england_2021.csv    524       116      ✅
                  df_england_2122.csv    538       116      ✅
                  df_england_2223.csv    554       116      ✅
                  df_england_2324.csv    570       116      ✅
                  df_england_2425.csv    562       116      ✅
df_england_2526_snapshot_20260326.csv    521       117      ✅
                   df_france_2021.csv    573       116      ✅
                   df_france_2122.csv    591       116      ✅
                   df_france_2223.csv    586       116      ✅
                   df_france_2324.csv    525       116      ✅
                   df_france_2425.csv    542       116      ✅
 df_france_2526_snapshot_20260326.csv    529       117      ✅
                  df_germany_2021.csv    496       116      ✅
                  df_germany_2122.csv    511       116      ✅
                  df_germany_2223.csv    506